In [9]:
%load_ext autoreload
%autoreload 2
import os
import sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = '1'
import numpy as np
import pandas as pd
from typing import Dict, List
# from dotenv import load_dotenv  # pip install python-dotenv
from dynclim.features.safety import SegmentationConfig, ThresholdConfig, CompositeConfig 
from dynclim.experiments.runner import ExperimentRunner
from dynclim.data import ASRSDataPipeline
from dynclim.utils import loadDomainsFromYaml
import arrow
# from dynclim.data.database import get_reports_from_db, create_database_engine
# load_dotenv(dotenv_path='/dynClimToolkit/.env')
# load_dotenv()
from pathlib import Path
# If your notebook is in the project root
# project_root = Path("/Users/mrosen44/Documents/dev/DynamicNarrativeClimateToolkit")
project_root = Path("/Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit")
# Option A — explicit reload of the package init
import importlib, dynclim.data
importlib.reload(dynclim.data)

from dynclim.data import (
    ASRSRawPipeline, ASRSRawConfig,
    AIDSDataPipeline, AIDSConfig,
    ASRSDataPipeline, ASRSConfig
)
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import logging
logging.getLogger().setLevel(logging.INFO)

# 1. Combine the latest ASRS raw downloads
asrs = ASRSRawPipeline(
    raw_dir = "/Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit/dynclim/data/raw/aviation/asrs_raw",
    output  = "/Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit/dynclim/data/processed/aviation/asrs_combined.parquet",
)
asrs.run(save=True)
print(asrs.summary())

# 2. Refresh AIDS event data
aids = AIDSDataPipeline(
    base_dir   = "/Users/michaelrosen/Documents/data_anlaysis/dyn_nar_clim_mv_analysis/data/aviation/aids/raw",
    output_dir = "/Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit/dynclim/data/processed/aviation",
)
aids_df = aids.run(save=True)
print(aids.summary(aids_df))

# 3. Push ASRS through the standardizer (ATC-only filter for feature extraction)
av = ASRSDataPipeline(
    events_path = "/Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit/dynclim/data/processed/aviation/asrs_combined.parquet",
    output_dir  = "/Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit/dynclim/data/processed/aviation",
    config      = ASRSConfig(atc_filter="all_atc"),
)
events_df = av.load_events(clean=True)
av.save(events_df, "events")        # → events.parquet (input to feature extraction)
print(av.summary(events_df))

2026-05-01 10:01:29,455 - INFO - Found 64 ASRS download files
2026-05-01 10:01:29,599 - INFO -   ASRS_DBOnline_1988.xls: 4,222 rows
2026-05-01 10:01:29,700 - INFO -   ASRS_DBOnline_1989-1.xls: 2,982 rows
2026-05-01 10:01:29,818 - INFO -   ASRS_DBOnline_1989-2.xls: 3,630 rows
2026-05-01 10:01:29,958 - INFO -   ASRS_DBOnline_1990-1.xls: 4,182 rows
2026-05-01 10:01:30,072 - INFO -   ASRS_DBOnline_1990-2.xls: 3,478 rows
2026-05-01 10:01:30,179 - INFO -   ASRS_DBOnline_1991-1.xls: 3,215 rows
2026-05-01 10:01:30,297 - INFO -   ASRS_DBOnline_1991-2.xls: 3,611 rows
2026-05-01 10:01:30,416 - INFO -   ASRS_DBOnline_1992-1.xls: 3,617 rows
2026-05-01 10:01:30,508 - INFO -   ASRS_DBOnline_1992-2.xls: 2,805 rows
2026-05-01 10:01:30,611 - INFO -   ASRS_DBOnline_1993-1.xls: 3,233 rows
2026-05-01 10:01:30,718 - INFO -   ASRS_DBOnline_1993-2.xls: 3,367 rows
2026-05-01 10:01:30,822 - INFO -   ASRS_DBOnline_1994-1.xls: 3,306 rows
2026-05-01 10:01:30,928 - INFO -   ASRS_DBOnline_1994-2.xls: 3,220 rows
2026

ASRS Raw Pipeline Summary
Total reports (post-dedup): 222,339
Duplicate ACNs removed:     2,550
Reports with combined narrative: 222,336 (100.0%)
Year span: 1988 – 2025
Reports per decade:
  1980s: 10,834
  1990s: 74,608
  2000s: 57,235
  2010s: 48,657
  2020s: 30,998
Months in range: 456
  missing:       0
  thin (likely incomplete): 0
ATC reporters (any tier): 21,601 (9.7%)


2026-05-01 10:01:45,210 - INFO -   Loaded Apre1975.zip: 8,850 rows
Skipping line 11147: '	' expected after '"'
Skipping line 17017: '	' expected after '"'
Skipping line 17637: '	' expected after '"'
Skipping line 17778: '	' expected after '"'
Skipping line 25050: '	' expected after '"'
Skipping line 30034: '	' expected after '"'
Skipping line 31250: '	' expected after '"'
Skipping line 31475: '	' expected after '"'
2026-05-01 10:01:46,652 - INFO -   Loaded a1975_79.zip: 31,715 rows
Skipping line 22802: '	' expected after '"'
Skipping line 23912: '	' expected after '"'
2026-05-01 10:01:48,441 - INFO -   Loaded a1980_84.zip: 35,066 rows
2026-05-01 10:01:49,965 - INFO -   Loaded a1985_89.zip: 31,205 rows
Skipping line 21392: '	' expected after '"'
2026-05-01 10:01:51,331 - INFO -   Loaded a1990_94.zip: 27,909 rows
Skipping line 9253: '	' expected after '"'
Skipping line 421: Expected 180 fields in line 421, saw 181
Skipping line 1432: Expected 180 fields in line 1432, saw 181
Skipping lin

FAA AIDS Summary
Total events: 232,635
Date range: 1945-07-23 -- 2026-04-12
Events by type:
  accident: 121,304
  incident: 111,331
Events by decade:
  1940s: 3
  1960s: 1
  1970s: 40,561
  1980s: 66,271
  1990s: 50,772
  2000s: 35,302
  2010s: 26,105
  2020s: 13,620
On-airport: 47.7%
Unique airport_ids (events): 10,480
Total fatalities: 35,030
Total injuries:   47,975


2026-05-01 10:02:01,113 - INFO - Loaded 222,339 rows, 127 columns
2026-05-01 10:02:01,113 - INFO - Column 'date': using 'time_date' (primary 'date' not found)
2026-05-01 10:02:01,113 - INFO - Resolved columns: {'acn': 'acn', 'date': 'time_date', 'narrative': 'cmbd_narrative', 'operator': 'aircraft_1aircraft_operator', 'airport': 'place_locale_reference', 'reporter_function': 'person_1function'}
2026-05-01 10:02:02,089 - WARNING - 7 events have unparseable dates
2026-05-01 10:02:17,232 - INFO - ATC filter ('all_atc'): 222,339 -> 21,601 rows (90.3% dropped)
2026-05-01 10:02:17,811 - WARNING - 1 events have invalid/missing dates
2026-05-01 10:02:17,835 - INFO - Cleaning: 222,339 -> 21,601 events (200,738 removed)
2026-05-01 10:02:18,625 - INFO - Saved 21,601 events to /Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit/dynclim/data/processed/aviation/events.parquet


ASRS Aviation Event Data Summary
Total events: 21,601
Date range: 1988-01-01 to 2025-12-01
Events with valid dates: 21,600 / 21,601
Unique operators: 40
Top operators:
  Air Carrier: 9,388
  : 5,620
  Personal: 2,620
  Corporate: 1,405
  Military: 1,215
Unique airports (primary): 2,658
Top airports:
  ZZZ.Airport: 931
  ZZZ.ARTCC: 592
  ZZZ.TRACON: 474
  SCT.TRACON: 212
  ZJX.ARTCC: 196
Unique reporter functions: 328
Narrative word count: mean=267, median=218, min=8, max=3336


In [11]:
# prepare domains and data for feature extraction
domain_definitions = loadDomainsFromYaml(project_root / 'config/domains/safety_domains.yaml')

# clean and prep reports for feature generation
pipeline = ASRSDataPipeline(
    events_path=project_root / "dynclim/data/processed/aviation/asrs_combined.parquet",
    output_dir=project_root / "dynclim/data/processed/aviation"
)
pipeline.config.atc_filter = "all_atc" # only uses atc reporters
results = pipeline.run()
report_data = pipeline.get_report_data()  # [(eid, narrative), ...]

2026-05-01 10:02:26,138 - INFO - Loading ASRS events from /Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit/dynclim/data/processed/aviation/asrs_combined.parquet
2026-05-01 10:02:26,793 - INFO - Loaded 222,339 rows, 127 columns
2026-05-01 10:02:26,793 - INFO - Column 'date': using 'time_date' (primary 'date' not found)
2026-05-01 10:02:26,794 - INFO - Resolved columns: {'acn': 'acn', 'date': 'time_date', 'narrative': 'cmbd_narrative', 'operator': 'aircraft_1aircraft_operator', 'airport': 'place_locale_reference', 'reporter_function': 'person_1function'}
2026-05-01 10:02:27,759 - WARNING - 7 events have unparseable dates
2026-05-01 10:02:42,982 - INFO - ATC filter ('all_atc'): 222,339 -> 21,601 rows (90.3% dropped)
2026-05-01 10:02:43,446 - WARNING - 1 events have invalid/missing dates
2026-05-01 10:02:43,470 - INFO - Cleaning: 222,339 -> 21,601 events (200,738 removed)
2026-05-01 10:02:44,286 - INFO - Saved 21,601 events to /Users/michaelrosen/Documents/dev/DynamicNarrat

In [12]:
# Run experiment with checkpointing

runner = ExperimentRunner(
    domain_definitions = domain_definitions,
    use_acceleration=True,
    device_pref='auto',
    use_parallel=False,
    embedding_model_names=[      
                                 
        'all-MiniLM-L6-v2',        # small, from sentence transformer team
        # 'all-mpnet-base-v2',     # medium, from sentence transformer team
        'all-roberta-large-v1',  # large, from sentence transformer team
        # 'BAAI/bge-large-en-v1.5' # very large, and generally better performance, but has different training method
                                   # when compared to above models which vary just on size
        # 'mixedbread-ai/mxbai-embed-large-v1',   # mid-sized
        'BAAI/bge-m3'                           # successor to BAAI model above
        ]
)

# Test with subset of configurations
config_filter = {
    #'segmentation_methods': ['sentence'],
    #'threshold_methods': ['top_k'],
    #'composite_methods': ['attention_weight', 'weighted_average'],
    'sentiment_methods': ['vader','roberta','siebert'] #['roberta', 'distilbert', 'finbert','vader','siebert']
}

results_df = runner.run_experiment(
    report_data, 
    experiment_name = "asrs_05-01-2026",
    config_filter=config_filter,
    resume_from_checkpoint=True,
    # save_interval=2,
    chunk_size=None
    # max_workers=10
)

   Registry: dropped 25 constant columns → 9 remaining (constants in registry_constants.json)

🔬 Running experiment 'asrs_05-01-2026'
   Total configurations: 135
   Pending configurations: 135
   Reports: 21601
   Embedding models: 3
   Parallel processing: False
   Caching: Enabled (embeddings=True, segments=True)


Configurations:   0%|          | 0/135 [00:00<?, ?it/s]


🔄 Initializing processor:
   Embedding model: BAAI/bge-m3
   Sentiment method: roberta
🚀 Using Apple Silicon MPS acceleration


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
2026-05-01 10:03:13,737 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Initialized Processor:
   Domains: ['LSC', 'SC', 'ST', 'CSP', 'SEH', 'SI', 'SR']
   Total items: 30
   Device: mps
   Caching: embeddings=True, segments=True
   Injected sentiment cache: 0 entries

🚀 Using batch-optimized processing...
   Pre-segmenting all texts...
[seg] SentenceSegmenter backend=spacy, min_length=5, unit=chars
[seg] built segmenter SentenceSegmenter for key=('5a5a91d711baaba23988779567c0b9f2', 'chars')
   Found 343838 unique segments
   Pre-encoding unique segments...
   ✅ Pre-cached 343838 embeddings

🚀 Starting processing...
   Reports: 21601
   Parallel: False


... (282 outputs left out)

Configurations:  99%|█████████▉| 134/135 [16:29:48<02:21, 141.57s/it, cache_hit=99.0%, time=145.2s]


💾 Checkpoint saved: 134 configurations
   Full results directory: checkpoints/asrs_05-01-2026/_cfg

🚀 Using batch-optimized processing...
   Pre-segmenting all texts...
   Found 343838 unique segments
   Pre-encoding unique segments...
   ✅ Pre-cached 343838 embeddings

🚀 Starting processing...
   Reports: 21601
   Parallel: False
✅ Processing complete!
   Total time: 46.57 seconds
   Rate: 463.9 reports/second


Configurations: 100%|██████████| 135/135 [16:30:36<00:00, 440.27s/it, cache_hit=99.1%, time=47.9s] 



💾 Checkpoint saved: 135 configurations
   Full results directory: checkpoints/asrs_05-01-2026/_cfg

💾 Checkpoint saved: 135 configurations
   Full results directory: checkpoints/asrs_05-01-2026/_cfg

✅ Experiment complete!
   Total time: 59436.7 seconds
   Average time per config: 440.27 seconds

📊 Final Cache Statistics:

📊 Cache Performance Report
Device: mps

🎯 Hit Rates:
   Segment Embeddings: 98.9%
   Sentiments: 100.0%
   Text Segmentation: 98.9%
   Similarities: 0.0%
   Overall: 99.1%

📈 Reuse Statistics:
   Unique segments: 343,495
   Total processed: 31,312,575
   Reuse factor: 91.16x

💾 Cache Sizes:
   segment_embeddings: 343,495 entries
   sentiments: 343,495 entries
   text_segments: 21,593 entries
   similarities: 0 entries
📄 Results exported to: checkpoints/asrs_05-01-2026_results.csv
📄 Results exported to: checkpoints/asrs_05-01-2026_results.h5

📁 Full report-level results saved in: checkpoints/asrs_05-01-2026/_cfg
   Each configuration's full results available as separ